# Intro


In [8]:
greeting = "안녕하세요 👋 (hello in Korean!)"
unicode_codepts = [ord(x) for x in greeting]

print(f"String: {greeting} (length: {len(greeting)}")
print(f"Unicode: {unicode_codepts} (length: {len(unicode_codepts)})")

String: 안녕하세요 👋 (hello in Korean!) (length: 26
Unicode: [50504, 45397, 54616, 49464, 50836, 32, 128075, 32, 40, 104, 101, 108, 108, 111, 32, 105, 110, 32, 75, 111, 114, 101, 97, 110, 33, 41] (length: 26)


In [9]:
greeting.encode("utf-8")

b'\xec\x95\x88\xeb\x85\x95\xed\x95\x98\xec\x84\xb8\xec\x9a\x94 \xf0\x9f\x91\x8b (hello in Korean!)'

In [10]:
print(f"Bytes: {list(greeting.encode('utf-8'))}")
print(f"Length: {len(list(greeting.encode('utf-8')))}")

Bytes: [236, 149, 136, 235, 133, 149, 237, 149, 152, 236, 132, 184, 236, 154, 148, 32, 240, 159, 145, 139, 32, 40, 104, 101, 108, 108, 111, 32, 105, 110, 32, 75, 111, 114, 101, 97, 110, 33, 41]
Length: 39


In [11]:
text = "Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception."
tokens = list(text.encode("utf-8"))
print("---")
print(text)
print(f"Length: {len(text)}")
print("---")
print(tokens)
print(f"Length: {len(tokens)}")

---
Ｕｎｉｃｏｄｅ! 🅤🅝🅘🅒🅞🅓🅔‽ 🇺‌🇳‌🇮‌🇨‌🇴‌🇩‌🇪! 😄 The very name strikes fear and awe into the hearts of programmers worldwide. We all know we ought to “support Unicode” in our software (whatever that means—like using wchar_t for all the strings, right?). But Unicode can be abstruse, and diving into the thousand-page Unicode Standard plus its dozens of supplementary annexes, reports, and notes can be more than a little intimidating. I don’t blame programmers for still finding the whole thing mysterious, even 30 years after Unicode’s inception.
Length: 533
---
[239, 188, 181, 239, 189, 142, 239, 189, 137, 239, 189, 131, 239, 189, 143, 239, 189, 132, 239, 189, 133, 33, 32, 240, 159, 133, 164, 240, 159, 133, 157, 240, 159, 133, 152, 240, 159, 133, 146, 240, 159, 133, 158, 240, 159, 133, 147, 240, 159, 133, 148, 226, 128, 189, 32, 240, 159, 135, 186, 226, 128, 140, 240, 159, 135, 179, 226, 128, 140, 240, 159, 135, 174, 226, 128, 140, 240, 159, 135, 168, 226, 128, 140, 240, 159, 135, 180, 226, 128, 140

Notice that the length of the list of bytes is greater than that of the original text since utf-8 encodes Unicode code points to byte streams of size 1 to 4.


# Getting started with BPE


In [12]:
from collections import Counter
from itertools import pairwise

In [13]:
def get_counts(tokens):
    return Counter(pairwise(tokens))


def merge(pair, idx, tokens):
    out = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            out.append(idx)
            i += 2
        else:
            out.append(tokens[i])
            i += 1
    return out

In [ ]:
with open("blog_unicode.txt") as f:
    text = f.read()
tokens = list(text.encode("utf-8"))

vocab_size = 276
num_merges = vocab_size - 256

merges = {}
out = tokens
for i in range(num_merges):
    pairs_to_counts = get_counts(out)
    pair = max(pairs_to_counts, key=pairs_to_counts.get)
    idx = 256 + i
    print(f"Merging {pair} into a new token {idx}")
    out = merge(pair, idx, out)
    # Keeping track of merges for decoding later on
    merges[pair] = idx

Merging (101, 32) into a new token 256
Merging (105, 110) into a new token 257
Merging (115, 32) into a new token 258
Merging (116, 104) into a new token 259
Merging (101, 114) into a new token 260
Merging (99, 111) into a new token 261
Merging (116, 32) into a new token 262
Merging (226, 128) into a new token 263
Merging (44, 32) into a new token 264
Merging (97, 110) into a new token 265
Merging (111, 114) into a new token 266
Merging (100, 32) into a new token 267
Merging (97, 114) into a new token 268
Merging (101, 110) into a new token 269
Merging (257, 103) into a new token 270
Merging (261, 100) into a new token 271
Merging (121, 32) into a new token 272
Merging (46, 32) into a new token 273
Merging (97, 108) into a new token 274
Merging (259, 256) into a new token 275


In [19]:
print("tokens length (pre-tokenization):", len(tokens))
print("out length (tokens post-tokenization):", len(out))
print(f"compression ratio: {len(tokens) / len(out):.2f}")

tokens length (pre-tokenization): 24636
out length (tokens post-tokenization): 19484
compression ratio: 1.26


# Decoding and encoding


In [27]:
vocab = {idx: bytes([idx]) for idx in range(256)}
# This works since merges preserves order of merging
for (p0, p1), idx in merges.items():
    # Concatenation since bytes objects
    vocab[idx] = vocab[p0] + vocab[p1]


def decode(ids):
    ids = b"".join([vocab[idx] for idx in ids])
    text = ids.decode("utf-8", errors="replace")
    return text


def encode(text):
    tokens = list(text.encode("utf-8"))  # tokens is list of integers now
    # Can do this since merges preserves order of merging
    for pair, idx in merges.items():
        tokens = merge(pair, idx, tokens)
    return tokens

In [28]:
print(decode(encode("hello world")))

hello world


In [29]:
text2 = decode(encode(text))
print(text2 == text)

True


In [30]:
valtext = "Many common characters, including numerals, punctuation, and other symbols, are unified within the standard and are not treated as specific to any given writing system. Unicode encodes thousands of emoji, with the continued development thereof conducted by the Consortium as a part of the standard.[4] Moreover, the widespread adoption of Unicode was in large part responsible for the initial popularization of emoji outside of Japan. Unicode is ultimately capable of encoding more than 1.1 million characters."
valtext2 = decode(encode(valtext))
print(valtext2 == valtext)

True


Note that while encode -> decode gives back the original text, this isn't necessarily the case for decode -> encode since some byte sequences may be invalid by utf-8 standards.


# GPT-2 and GPT-4 tokenizers


### Forcing smarter merges with regex splits


In [36]:
import regex as re

In [37]:
gpt2pat = re.compile(
    r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
)

print(gpt2pat.findall("Hello've world123 HOW'S are        you?!?!   "))

['Hello', "'ve", ' world', '123', ' HOW', "'", 'S', ' are', '       ', ' you', '?!?!', '   ']


Note that training the GPT-2 tokenizer likely involved other rules/techniques in addition to chunking up the text with the regex pattern above and running BPE on the resultant list.


### Tiktoken


In [40]:
import tiktoken

In [42]:
# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    hello world!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    hello world!!!"))

[220, 220, 220, 23748, 995, 10185]
[262, 24748, 1917, 12340]
